# Bloque I — Python, pandas y análisis descriptivo
## Notebook Entregable — Ejercicio Integrador

**Dataset:** `ventas_mayo_2026.csv`  
**Tareas:**
1. Ventas totales por canal
2. Ticket medio por región
3. Categoría con mayor número de unidades vendidas
4. Gráfico de barras con ventas por canal
5. Tres conclusiones de negocio

In [ ]:
# Configuración común
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## Carga y preparación de datos

Antes de abordar el ejercicio, cargamos el dataset y aplicamos la misma limpieza vista en clase para asegurar que los datos son correctos.

In [ ]:
df = pd.read_csv('data/ventas_mayo_2026.csv')
print('Shape original:', df.shape)
df.head()

In [ ]:
# --- Limpieza ---
df_limpio = df.copy()

# Eliminar duplicados
df_limpio = df_limpio.drop_duplicates()
print(f'Duplicados eliminados: {df.shape[0] - df_limpio.shape[0]}')

# Convertir fecha
df_limpio['fecha'] = pd.to_datetime(df_limpio['fecha'])

# Imputar nulos
df_limpio['precio_unitario'] = df_limpio['precio_unitario'].fillna(
    df_limpio['precio_unitario'].median()
)
df_limpio['region'] = df_limpio['region'].fillna('Sin informar')

# Variables derivadas
df_limpio['mes']            = df_limpio['fecha'].dt.month
df_limpio['dia_semana']     = df_limpio['fecha'].dt.day_name()
df_limpio['importe_con_iva']= df_limpio['importe'] * 1.21
df_limpio['ticket_unitario']= df_limpio['importe'] / df_limpio['unidades']

print(f'Shape limpio: {df_limpio.shape}')
print(f'Nulos restantes:\n{df_limpio.isnull().sum()}')

---
## Tarea 1 — Ventas totales por canal

Agrupamos por la columna `canal` y sumamos el importe para saber cuánto factura cada canal de venta.

In [ ]:
ventas_por_canal = (
    df_limpio
    .groupby('canal')
    .agg(
        ventas_totales  = ('importe', 'sum'),
        importe_medio   = ('importe', 'mean'),
        operaciones     = ('cliente_id', 'count'),
        unidades_totales= ('unidades', 'sum')
    )
    .sort_values('ventas_totales', ascending=False)
)

ventas_por_canal

---
## Tarea 2 — Ticket medio por región

El ticket medio indica el importe promedio que deja cada transacción en cada región. Un ticket elevado puede señalar mayor poder adquisitivo o preferencia por productos de mayor precio.

In [ ]:
ticket_por_region = (
    df_limpio
    .groupby('region')
    .agg(
        ticket_medio    = ('importe', 'mean'),
        ventas_totales  = ('importe', 'sum'),
        operaciones     = ('cliente_id', 'count')
    )
    .sort_values('ticket_medio', ascending=False)
)

ticket_por_region

---
## Tarea 3 — Categoría con mayor número de unidades vendidas

El volumen de unidades nos habla de la popularidad de cada categoría, independientemente del precio. Una categoría puede tener mucho volumen pero poco importe si sus productos son baratos.

In [ ]:
unidades_por_categoria = (
    df_limpio
    .groupby('categoria')
    .agg(
        unidades_totales = ('unidades', 'sum'),
        importe_total    = ('importe', 'sum'),
        ticket_medio     = ('importe', 'mean')
    )
    .sort_values('unidades_totales', ascending=False)
)

categoria_top_unidades = unidades_por_categoria.index[0]
print(f'Categoría con más unidades vendidas: {categoria_top_unidades}')
print(f'Total unidades: {unidades_por_categoria.loc[categoria_top_unidades, "unidades_totales"]:,}')
print()
unidades_por_categoria

---
## Tarea 4 — Gráfico de barras: ventas por canal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Gráfico 1: Ventas totales por canal ---
colores = ['#2196F3', '#4CAF50', '#FF9800']
bars = axes[0].bar(
    ventas_por_canal.index,
    ventas_por_canal['ventas_totales'],
    color=colores,
    edgecolor='white',
    linewidth=1.2
)
axes[0].set_title('Ventas totales por canal', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Canal')
axes[0].set_ylabel('Importe total (€)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Etiquetas encima de cada barra
for bar in bars:
    h = bar.get_height()
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        h * 1.01,
        f'{h:,.0f} €',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

# --- Gráfico 2: Número de operaciones por canal ---
axes[1].bar(
    ventas_por_canal.index,
    ventas_por_canal['operaciones'],
    color=colores,
    edgecolor='white',
    linewidth=1.2
)
axes[1].set_title('Número de operaciones por canal', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Canal')
axes[1].set_ylabel('Número de operaciones')

for bar in axes[1].patches:
    h = bar.get_height()
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        h * 1.01,
        f'{int(h):,}',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('data/ventas_por_canal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado.')

---
## Gráfico complementario — Ticket medio por región

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

colores_region = ['#E91E63', '#9C27B0', '#3F51B5', '#009688', '#FF5722']
bars = ax.barh(
    ticket_por_region.index,
    ticket_por_region['ticket_medio'],
    color=colores_region,
    edgecolor='white',
    linewidth=1.2
)

ax.set_title('Ticket medio por región', fontsize=13, fontweight='bold')
ax.set_xlabel('Importe medio por operación (€)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

for bar in bars:
    w = bar.get_width()
    ax.text(
        w * 1.005, bar.get_y() + bar.get_height() / 2,
        f'{w:,.0f} €',
        va='center', ha='left', fontsize=9
    )

plt.tight_layout()
plt.show()

---
## Tarea 5 — Conclusiones de negocio

In [ ]:
canal_top          = ventas_por_canal.index[0]
canal_top_importe  = ventas_por_canal.loc[canal_top, 'ventas_totales']
pct_canal_top      = canal_top_importe / ventas_por_canal['ventas_totales'].sum() * 100

region_ticket_top  = ticket_por_region.index[0]
ticket_top_valor   = ticket_por_region.loc[region_ticket_top, 'ticket_medio']
region_ticket_low  = ticket_por_region.index[-1]
ticket_low_valor   = ticket_por_region.loc[region_ticket_low, 'ticket_medio']

cat_unid_top       = unidades_por_categoria.index[0]
cat_unid_valor     = unidades_por_categoria.loc[cat_unid_top, 'unidades_totales']
cat_importe_top    = unidades_por_categoria.sort_values('importe_total', ascending=False).index[0]

print('=' * 65)
print('CONCLUSIONES DE NEGOCIO — Mayo 2026')
print('=' * 65)
print()
print(f'1. CANAL DOMINANTE: "{canal_top}" concentra el {pct_canal_top:.1f} % de la')
print(f'   facturación total ({canal_top_importe:,.0f} €). Reforzar la')
print(f'   experiencia en este canal y asegurar su disponibilidad es')
print(f'   prioritario para mantener el volumen de ingresos.')
print()
print(f'2. DISPARIDAD REGIONAL EN TICKET: La región "{region_ticket_top}"')
print(f'   presenta el ticket medio más alto ({ticket_top_valor:,.0f} €), mientras')
print(f'   que "{region_ticket_low}" tiene el más bajo ({ticket_low_valor:,.0f} €).')
print(f'   Esta brecha ({ticket_top_valor - ticket_low_valor:,.0f} €) sugiere perfiles de cliente')
print(f'   muy distintos: se recomienda adaptar el catálogo y las')
print(f'   promociones a cada región.')
print()
print(f'3. VOLUMEN VS VALOR EN CATEGORÍAS: "{cat_unid_top}" lidera en')
print(f'   unidades vendidas ({cat_unid_valor:,} uds.), pero la categoría con')
print(f'   mayor importe total es "{cat_importe_top}". Esto indica que')
print(f'   "{cat_unid_top}" mueve mucho stock a precios bajos, mientras')
print(f'   que "{cat_importe_top}" genera más valor por operación.')
print(f'   La estrategia de margen debe diferenciarse entre ambas.')
print()
print('=' * 65)